# Day 035 — Exercise 2: extract_info

**What you'll build:** `extract_info(doc, model) -> dict` — use Pydantic + `format='json'` to extract `ArticleInfo` (title, summary, sentiment, key_points) from a document dict. Returns an error envelope `{status, info, error}` — never raises.

**Why it matters:** Schema-guided extraction (Day 4/9 pattern) applied to a pipeline stage. The error envelope means the batch layer never has to catch exceptions — it just checks the `status` key.

## Provided: ArticleInfo Pydantic Model

In [ ]:
import json
import ollama
from pydantic import BaseModel, Field

class ArticleInfo(BaseModel):
    title:      str       = Field(description='Topic or title in 3-6 words')
    summary:    str       = Field(description='One sentence summary')
    sentiment:  str       = Field(description='positive, negative, or neutral')
    key_points: list[str] = Field(default_factory=list,
                                  description='Up to 3 key points as short phrases')

## Provided: fetch_text (to create test docs)

In [ ]:
import requests
from pathlib import Path

def fetch_text(source: str) -> dict:
    src  = str(source)
    text = None
    kind = 'text'

    if src.startswith('http://') or src.startswith('https://'):
        try:
            response = requests.get(src, timeout=10)
            response.raise_for_status()
            text = response.text
            kind = 'url'
        except Exception as e:
            text = '[fetch error: ' + str(e) + ']'
            kind = 'url_error'
    else:
        try:
            p = Path(src)
            if p.exists() and p.is_file():
                text = p.read_text(encoding='utf-8')
                kind = 'file'
        except Exception:
            pass

    if text is None:
        text = src
        kind = 'text'

    return {
        'source':     src,
        'kind':       kind,
        'content':    text,
        'char_count': len(text),
    }

## Your Implementation

In [ ]:
def extract_info(doc: dict, model: str = 'llama3.2') -> dict:
    """
    Extract structured info from a doc dict using ArticleInfo schema.

    Returns error envelope: always a dict with 'status', 'info', 'error'.
    On success:  {**doc, 'info': ArticleInfo.model_dump(), 'status': 'ok',    'error': None}
    On failure:  {**doc, 'info': None,                    'status': 'error', 'error': str(e)}
    """
    schema = ArticleInfo.model_json_schema()
    prompt = (
        'Extract information from the document below. '
        'Return valid JSON matching this schema:\n'
        + json.dumps(schema, indent=2)
        + '\n\nDocument:\n' + doc['content'][:1500]
    )
    # TODO: try:
    #     response = ollama.chat(model=model, messages=[...], format='json')
    #     info = ArticleInfo.model_validate_json(response['message']['content'])
    #     return {**doc, 'info': info.model_dump(), 'status': 'ok', 'error': None}
    # TODO: except Exception as e:
    #     return {**doc, 'info': None, 'status': 'error', 'error': str(e)}
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'extract_info' in globals()
        passed += 1; print('\u2705 Check 1: extract_info defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Pre-run: call extract_info once for all checks
    doc = fetch_text('Python is a popular high-level programming language known '
                     'for its clean syntax and large ecosystem.')
    result = None
    try:
        result = extract_info(doc)
    except Exception as e:
        print(f'\u274c extract_info raised unexpectedly: {e}')
        print('(extract_info should return an error envelope, not raise)')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: returns dict with status, info, error keys
    try:
        assert isinstance(result, dict), f'expected dict, got {type(result).__name__}'
        for k in ('source', 'kind', 'content', 'char_count', 'status', 'info', 'error'):
            assert k in result, f'missing key: {k}'
        passed += 1; print(f'\u2705 Check 2: result has all required keys')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: original doc keys preserved
    try:
        assert result['source']     == doc['source']
        assert result['content']    == doc['content']
        assert result['char_count'] == doc['char_count']
        passed += 1; print('\u2705 Check 3: original doc fields preserved')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: status is 'ok' or 'error' (never missing)
    try:
        assert result['status'] in ('ok', 'error'), \
            f"status must be 'ok' or 'error', got {result['status']!r}"
        passed += 1; print(f"\u2705 Check 4: status={result['status']!r}")
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: if ok, info is a dict with all ArticleInfo fields
    try:
        if result['status'] == 'ok':
            info = result['info']
            assert isinstance(info, dict), f'info should be dict, got {type(info).__name__}'
            for k in ('title', 'summary', 'sentiment', 'key_points'):
                assert k in info, f'info missing field: {k}'
            assert isinstance(info['key_points'], list), \
                f'key_points should be list, got {type(info["key_points"]).__name__}'
            passed += 1; print(f"\u2705 Check 5: info dict has title={info['title']!r}, "
                               f"sentiment={info['sentiment']!r}")
        else:
            assert result['error'] is not None, \
                'status=error but error field is None'
            passed += 1; print(f"\u2705 Check 5: status=error, error={result['error']!r}")
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import json
import ollama
from pydantic import BaseModel, Field

class ArticleInfo(BaseModel):
    title:      str       = Field(description='Topic or title in 3-6 words')
    summary:    str       = Field(description='One sentence summary')
    sentiment:  str       = Field(description='positive, negative, or neutral')
    key_points: list[str] = Field(default_factory=list,
                                  description='Up to 3 key points as short phrases')

def extract_info(doc: dict, model: str = 'llama3.2') -> dict:
    schema = ArticleInfo.model_json_schema()
    prompt = (
        'Extract information from the document below. '
        'Return valid JSON matching this schema:\n'
        + json.dumps(schema, indent=2)
        + '\n\nDocument:\n' + doc['content'][:1500]
    )
    try:
        response = ollama.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            format='json',
        )
        info = ArticleInfo.model_validate_json(response['message']['content'])
        return {**doc, 'info': info.model_dump(), 'status': 'ok',    'error': None}
    except Exception as e:
        return {**doc, 'info': None,               'status': 'error', 'error': str(e)}
```

</details>